In [ ]:
# %run take_split.py --target_dataset loris3/tulu3-test-small --test 100 --train 1000

In [8]:
!top

=top - 15:01:10 up 105 days, 23:59,  0 users,  load average: 14.91, 13.39, 12.32
Tasks: 1065 total,  11 running, 1050 sleeping,   4 stopped,   0 zombie
%Cpu(s): 11.3 us,  5.1 sy,  0.0 ni, 83.5 id,  0.0 wa,  0.0 hi,  0.1 si,  0.0 st
MiB Mem : 515844.7 total,  45103.0 free,  37915.7 used, 432826.0 buff/cache
MiB Swap:      0.0 total,      0.0 free,      0.0 used. 235735.1 avail Mem 

    PID USER      PR  NI    VIRT    RES    SHR S  %CPU  %MEM     TIME+ COMMAND  
1556073 loriss2+  20   0   40.4g   3.6g 618988 R 100.0   0.7  54:35.17 python3  
1569594 loriss2+  20   0   39.3g   3.4g 622492 S 100.0   0.7  43:11.45 python3  
1508496 loriss2+  20   0   73.9g   5.0g 677652 S  94.7   1.0 147:12.89 python3  
1556017 loriss2+  20   0   41.1g   7.3g   5.0g R  94.7   1.4  54:19.31 python3  
1560601 nobody    20   0  618.0g   2.4g   1.3g S  94.7   0.5  57:44.17 python   
1556209 loriss2+  20   0   57.3g   6.5g   3.5g R  89.5   1.3  54:16.58 python3  
2163216 loriss2+  20   0  879824 223540  51984 

In [99]:
!du -h /tmp/cache/scoring

4.6M	/tmp/cache/scoring/scoring/partial/LESSEstimator: normalize=True/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42/loris3/tulu-3-sft-olmo-2-mixture-0225-sample/train/loris3/tulu-3-sft-olmo-2-mixture-0225-sample/test/partial/MSECoderProjUSimpSparse/1 random examples with seed 6
12M	/tmp/cache/scoring/scoring/partial/LESSEstimator: normalize=True/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42/loris3/tulu-3-sft-olmo-2-mixture-0225-sample/train/loris3/tulu-3-sft-olmo-2-mixture-0225-sample/test/partial/MSECoderProjUSimpSparse/1 random examples with seed 10
12M	/tmp/cache/scoring/scoring/partial/LESSEstimator: normalize=True/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42/loris3/tulu-3-sft-olmo-2-mixture-0225-sample/train/loris3/tulu-3-sft-olmo-2-mixture-0225-sample/test/partial/MSECoderProjUSimpSparse/1 random examples with seed 9
12M	/tmp/cache/scoring/scoring/partial/LESSEstimator: normalize=True/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e

In [65]:
ls -lhas cache/scoring/full/full.parquet

29M -rw-r--r-- 1 loriss21cs dm 29M Sep 26 16:20 cache/scoring/full/full.parquet


In [59]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:           503Gi        34Gi        66Gi       235Gi       402Gi       230Gi
Swap:             0B          0B          0B


In [ ]:
# import os
# import torch
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# from transformers import AutoTokenizer, AutoModelForCausalLM


# tokenizer = AutoTokenizer.from_pretrained("models/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged")
# model = AutoModelForCausalLM.from_pretrained("models/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged")
# messages = [
#     {"role": "user", "content": "could you tell us some of your political beliefs"},
# ]
# inputs = tokenizer.apply_chat_template(
# 	messages,
# 	add_generation_prompt=True,
# 	tokenize=True,
# 	return_dict=True,
# 	return_tensors="pt",
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# ).to(model.device)

# outputs = model.generate(**inputs, max_new_tokens=40)
# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
# !rm -rf cache/scoring/

In [ ]:
# import wandb


# wandb.login()


# api = wandb.Api()


# entity = "loriss"
# project = "linear_coder"


# runs = api.runs(f"{entity}/{project}")


# for run in runs:
#     print(f"Deleting run: {run.name} ({run.id})")
#     run.delete()


In [ ]:
import pandas as pd

df = pd.read_parquet("./cache/scoring/partial")
# assert (df.groupby(["explanation_type", "linear_coder"]).count() == n_test).all().all()


In [ ]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('pred_gain', 'mean')].sort_values()

In [ ]:
# df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('l1', 'mean')].sort_values()

In [ ]:
mean_mse = df.groupby(["explanation_type", "linear_coder", "estimator"])['mse'].mean().reset_index()
mean_mse['rank'] = mean_mse.groupby(['explanation_type', 'linear_coder'])['mse'].rank(method='min')
mean_mse = mean_mse.sort_values(['linear_coder', 'explanation_type', 'rank'])
mean_mse

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_palette("Set2")

mean_mse = df.groupby(["explanation_type", "linear_coder", "estimator"])['mse'].mean().reset_index()
mean_mse = mean_mse[~mean_mse["explanation_type"].str.contains("random")]



best_estimators = mean_mse.loc[mean_mse.groupby(['explanation_type', 'linear_coder'])['mse'].idxmin()]
plt.tight_layout()

estimators = best_estimators['estimator'].unique()
markers = ['o', 's', 'D', '^', 'v', '<', '>', 'P', 'X', '*']  
colors = sns.color_palette("Set2", n_colors=len(estimators)) 
marker_map = {est: markers[i % len(markers)] for i, est in enumerate(estimators)}
color_map = {est: colors[i % len(colors)] for i, est in enumerate(estimators)}

explanation_types = list(best_estimators['explanation_type'].unique())
linear_coders = list(best_estimators['linear_coder'].unique())

fig, ax = plt.subplots(figsize=(6,6))

for _, row in best_estimators.iterrows():
    x = linear_coders.index(row['linear_coder'])  
    y = explanation_types.index(row['explanation_type'])  
    ax.scatter(
        x, y,
        marker=marker_map[row['estimator']],
        color=color_map[row['estimator']],
        s=200
    )

ax.set_xticks(range(len(linear_coders)))
ax.set_xticklabels(linear_coders)
ax.set_yticks(range(len(explanation_types)))
ax.set_yticklabels(explanation_types)
ax.set_xlabel('Linear Coder') 
ax.set_ylabel('Explanation Type')  
ax.set_xticklabels(linear_coders, rotation=45, ha='right')  

for est in estimators:
    ax.scatter([], [], marker=marker_map[est], color=color_map[est], label=est, s=200)
ax.legend(title='Best Estimator', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()


In [ ]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('mse', 'mean')].sort_values()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
plt.tight_layout()

filtered_df = df.copy()  

estimators = filtered_df['estimator'].unique()
if len(estimators) != 2:
    raise ValueError("This plot assumes exactly 2 estimators.")

max_per_group = filtered_df.groupby('linear_coder')['pred_gain'].max().sort_values(ascending=False)
sorted_linear_coders = max_per_group.index.tolist()

g = sns.catplot(
    data=filtered_df,
    x="explanation_type",
    y="pred_gain",
    hue="estimator",
    col="linear_coder",
    col_order=sorted_linear_coders,
    kind="box",
    palette="Set2",
    dodge=True,



)

for ax in g.axes.flat:
    ax.axhline(1.0, color='black', linestyle='dashed')



g.set_xticklabels(rotation=90, ha='right')


g.fig.subplots_adjust(wspace=0.0) 


plt.show()
